# Assessment 2
- Singh, Lovedeep - s8208559
- Gurung, Diya - s8221355
- Surinder, Surinder - s8247508

## Importing modules

> By default, pandas gives out values in scientific-notation which is like a little difficult to grasp so the last line here just displays the float data type as decimals with 3 digits after the decimal point instead of the scientific-notation

In [ ]:
import pandas as pd
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns

pd.options.display.float_format = '{:.3f}'.format

## Loading the Dataset

In [ ]:
df = pd.read_csv("./data/patronage.csv")

## Enforcing Types

The columns, `Year` and `Month` are string data types, because of this, if we later try to use them in numerical operations, they are prone to errors, so we convert them into `integer` 

In [ ]:
df["Year"] = df["Year"].astype(int)
df["Month"] = df["Month"].astype(int)

## Adding Features

We are supposed to add the folowing features.
- `MonthStart`: A date for the first of the month
- `Period`: groups years into three COVID-era buckets, *Pre-2020*, *2020-2021*, *Post-2021*
- `Is_Weekday`: Simple true false for if the `Day_of_week` is Monday-to-Friday or Saturday-Sundary
- `3_month_mean`: Instead of giving raw data for a month, we give the average for 3 months

In [ ]:
df["MonthStart"] = pd.to_datetime(dict(
    day=1,
    month=df["Month"],
    year=df["Year"],
))

# By default we set the Period to be "Post-2021"
# so that we only have to check for other two conditions and not all three
df["Period"] = "Post-2021"
df.loc[df["Year"] < 2020, "Period"] = "Pre-2020"
df.loc[(df["Year"] >= 2020) & (df["Year"] <= 2021), "Period"] = "2020-2021"

weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
df["Is_Weekday"] = df["Day_of_week"].isin(weekdays)

df = df.sort_values(["Mode", "Day_type", "MonthStart"])
df["3_month_mean"] = df.groupby(["Mode", "Day_type"])["Pax_daily"].transform(
    lambda s: s.rolling(3, min_periods=1).mean()
)

## Standardising category labels

In [ ]:
df["Mode"] = df["Mode"].str.strip().str.title()
df["Day_type"] = df["Day_type"].str.strip().str.title()
df["Day_of_week"] = df["Day_of_week"].str.strip().str.title()

## Handling Null and Duplicate values

After running `df.isnull().sum()` we get to see that in the provided dataset, there are NO null values, in any of the columns. But if there were null values, we can just remove them by doing, `df = df.dropna()`

After running `df.duplicated().sum()` we notice that there no duplicate rows at all (the dataset seems to be already cleaned). Buf if there were duplicate values, we can just remove them by doing, `df = df.drop_duplicates()`

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

## Handling Missing/Anomalous Pax_daily values

Since there are no null values in the entire dataset, even if there were, they were removed in the previous steps. So, at this point, there are no missing Pax_daily values.

Then we check if any of the values are negative, since a negative passenger count is physically impossible, if there are any negative values, we just remove them.

In [ ]:
print("Missing Pax_daily values:", df["Pax_daily"].isna().sum())
print("Negative values:", (df["Pax_daily"] < 0).sum())

df = df[df["Pax_daily"] >= 0]

## Exporting the Cleaned Dataset

We use the `to_csv()` method to export the dataset to any path.

In [ ]:
df.to_csv("data/patronage_clean.csv", index=False)

## Inferential tests

In [ ]:
# Split into weekday and weekend groups
weekday = df[df["Is_Weekday"] == True]["Pax_daily"]
weekend = df[df["Is_Weekday"] == False]["Pax_daily"]

t_stat, p_value = stats.ttest_ind(weekday, weekend)
print(f"Weekday mean: {weekday.mean():.2f}")
print(f"Weekend mean: {weekend.mean():.2f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Significant difference between weekday and weekend patronage")
else:
    print("Result: No significant difference found")

Weekday mean: 267415.07
Weekend mean: 146640.47
T-statistic: 14.8527
P-value: 0.0000
Result: Significant difference between weekday and weekend patronage


## Plotting

The following plot contains the following:

- A histogram of Pax_daily
- Kernal density of Pax_daily
- Q-Q plot of pax daily

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram
axes[0].hist(df["Pax_daily"], bins=30, edgecolor="black")
axes[0].set_title("Histogram of Pax_daily")
axes[0].set_xlabel("Daily Passengers")
axes[0].set_ylabel("Frequency")

# 2. Kernel Density
df["Pax_daily"].plot.kde(ax=axes[1])
axes[1].set_title("Kernel Density of Pax_daily")
axes[1].set_xlabel("Daily Passengers")

# 3. Q-Q Plot
(osm, osr), (slope, intercept, r) = stats.probplot(df["Pax_daily"])
axes[2].scatter(osm, osr, alpha=0.5, s=10)
axes[2].plot(osm, slope*osm + intercept, color="red", label="Normal line")
axes[2].set_title("Q-Q Plot of Pax_daily")
axes[2].set_xlabel("Theoretical Quantiles")
axes[2].set_ylabel("Sample Quantiles")
axes[2].legend()

plt.tight_layout()
plt.savefig("distribution_checks.png")
plt.show()

# Linear Regression Model

In [ ]:
df["MonthStart_num"] = (df["MonthStart"] - df["MonthStart"].min()).dt.days

In [ ]:
fig, axes = plt.subplots(len(df["Mode"].unique()), 1, figsize=(10, 20))

for i, mode in enumerate(df["Mode"].unique()):
    mode_df = df[df["Mode"] == mode].groupby("MonthStart_num")["Pax_daily"].mean().reset_index()
    
    X = mode_df["MonthStart_num"].values.reshape(-1, 1)
    y = mode_df["Pax_daily"].values
    
    model = sklearn.linear_model.LinearRegression()
    model.fit(X, y)
    
    trend = model.predict(X)
    slope = model.coef_[0]
    
    axes[i].scatter(mode_df["MonthStart_num"], y, alpha=0.5, s=10, label="Actual")
    axes[i].plot(mode_df["MonthStart_num"], trend, color="red", label=f"Trend (slope={slope:.2f})")
    axes[i].set_title(f"{mode} - Long Run Trend")
    axes[i].set_xlabel("Days since start")
    axes[i].set_ylabel("Avg Daily Passengers")
    axes[i].legend()

plt.tight_layout()
plt.savefig("linear_regression_trends.png")
plt.show()